# Week 07 — Routing, Memory & Tools
_Nawaloka Hospital AI Assistant — Foundation Concepts_

---

This notebook covers everything built in **Week 07**: the imperative orchestration pipeline, all four routing paths, and the complete 4-layer memory system. It is the **prerequisite** for Notebook 02 (LangGraph multi-agent system).

## What You Will Learn

| Section | Topic |
|---------|-------|
| **1** | Build the agent — wiring all components together |
| **2** | User identification from free-form chat |
| **3** | Routing engine — 4 routes: `direct`, `crm`, `rag`, `web_search` |
| **4** | Short-term memory — conversation ring buffer |
| **5** | Memory distillation — LLM extracts long-term facts |
| **6** | Long-term semantic recall — pgvector cosine search |
| **7** | Episodic memory — full conversation snapshots |
| **8** | Procedural memory — workflow guidance |
| **9** | Without vs. with memory — the impact of context |
| **10** | Multi-turn progressive conversation |
| **11** | LangFuse observability |

## Architecture (Week 07 — Imperative Pipeline)

```
User Message
     │
     ▼
AgentOrchestrator.chat()
     ├─ 1. _recall_memory()   → ST turns + LT facts  → memory_context (str)
     ├─ 2. router.route()     → RouteDecision (crm | rag | web_search | direct)
     ├─ 3. _dispatch()        → CRMTool | RAGTool | WebSearchTool → tool_output (str)
     ├─ 4. _synthesise()      → LLM merges memory + tool output   → final answer
     ├─ 5. _store_turns()     → ST write (Supabase ring buffer)
     └─ 6. _maybe_distill()   → LLM extracts LT facts (if triggered)
```

## Prerequisites
```bash
make seed-crm-xl        # Populate CRM with patients, doctors, bookings
make ingest-qdrant      # Ingest knowledge base into Qdrant (auto-runs on first start)
python scripts/seed_procedures.py  # Load procedural memory
```

---
## Section 0 — Setup

In [ ]:
import sys, os, re, time, json
sys.path.insert(0, "../src")

from dotenv import load_dotenv
load_dotenv()

from infrastructure.log import setup_logging
from loguru import logger
setup_logging("INFO", for_notebook=True)

import pandas as pd
from datetime import datetime
from sqlalchemy.orm import sessionmaker

from infrastructure.db import create_tables, get_sql_engine
from infrastructure.db.crm_models import Patient, Booking, Doctor, Location, Specialty
from infrastructure.llm import get_chat_llm, get_default_embeddings
from memory import (
    ConversationTurn, ShortTermMemoryStore, LongTermMemoryStore,
    EpisodicMemoryStore, ProceduralMemoryStore,
    MemoryDistiller, MemoryRecaller, create_episode_from_turns,
)
from infrastructure.config import ST_MAX_TURNS, ST_TTL_SECONDS

create_tables()
embedder   = get_default_embeddings()
llm        = get_chat_llm()
st_store   = ShortTermMemoryStore()
lt_store   = LongTermMemoryStore(embedder)
distiller  = MemoryDistiller(llm, lt_store)
recaller   = MemoryRecaller(st_store, lt_store)

logger.success("Environment ready")
logger.info(f"  LLM       : {getattr(llm, 'model_name', 'unknown')}")
logger.info(f"  ST store  : Supabase (st_turns)")
logger.info(f"  LT store  : Supabase pgvector")

---
## Section 1 — Build the Agent

`build_agent()` wires all components:
- **3 LLMs**: Router (`gpt-4o-mini`) · Extractor (`llama-3.1-8b-instant`) · Chat (`gemini-2.5-flash`)
- **3 Tools**: CRM (SQLAlchemy → Supabase) · RAG (CAG → CRAG → Qdrant) · Web (Tavily)
- **Memory**: ShortTermStore + LongTermStore + MemoryDistiller + MemoryRecaller
- **Observability**: LangFuse v3 traces every step

On first run, `ensure_kb_ingested()` checks if the Qdrant `nawaloka` collection has data — if not, it runs the ingestion pipeline automatically.

In [ ]:
from agents import build_agent, AgentResponse

agent = build_agent(enable_crm=True, enable_rag=True, enable_web=True)

logger.success("Agent built")
logger.info(f"  CRM tool   : {'✓' if agent.crm_tool else '✗'}")
logger.info(f"  RAG tool   : {'✓' if agent.rag_tool else '✗'}")
logger.info(f"  Web search : {'✓' if agent.web_tool else '✗'}")

# ── Visualize the LangGraph StateGraph ──────────────────────────
from IPython.display import Image, display

graph_png = agent.graph.get_graph(xray=True).draw_mermaid_png()
display(Image(graph_png))

---
## Section 2 — User Identification from Chat

In production, users interact via WhatsApp or a web chat. Their identity arrives through the **conversation itself** — not a hardcoded variable. The `extract_phone()` helper handles any Sri Lankan number format:

| User types | Normalised to |
|---|---|
| `+94 78 10 30 736` | `94781030736` |
| `0781030736` | `94781030736` |
| `+94-781-030-736` | `94781030736` |
| `(078) 103-0736` | `94781030736` |

In [ ]:
def extract_phone(text: str) -> str:
    """Extract and normalise a Sri Lankan phone number from free-form text."""
    match = re.search(r"\+?[\d][\d\s\-\.\(\)]{7,18}[\d]", text)
    if not match:
        raise ValueError("No phone number found in the message.")
    raw = re.sub(r"\D", "", match.group())
    if raw.startswith("0") and len(raw) == 10:
        raw = "94" + raw[1:]
    elif len(raw) == 9 and not raw.startswith("94"):
        raw = "94" + raw
    logger.info(f"  Normalised → {raw}")
    return raw


def show_response(resp: AgentResponse, label: str = "") -> None:
    """Pretty-print an AgentResponse."""
    print("=" * 72)
    if label:
        print(f"  {label}")
    print("=" * 72)
    print(f"  Route   : {resp.route}" + (f" / {resp.action}" if resp.action else ""))
    print(f"  Latency : {resp.latency_ms} ms")
    if resp.memory_context and resp.memory_context.strip():
        lines = resp.memory_context.strip().split("\n")
        print(f"\n  Memory context ({len(lines)} lines):")
        for ln in lines[:8]:
            print(f"    {ln}")
        if len(lines) > 8:
            print(f"    ... ({len(lines) - 8} more lines)")
    if resp.tool_output and resp.tool_output.strip():
        print(f"\n  Tool output:")
        for ln in resp.tool_output.strip().split("\n")[:6]:
            print(f"    {ln}")
    print(f"\n  Answer:\n    {resp.answer}")
    print("=" * 72)


def _crm_session():
    return sessionmaker(bind=get_sql_engine())()


def show_patient(user_id: str) -> None:
    session = _crm_session()
    try:
        p = session.query(Patient).filter(Patient.external_user_id == user_id).first()
        if not p:
            logger.warning(f"No patient for user_id={user_id}")
            return
        rows = [{"Field": k, "Value": v} for k, v in {
            "Name": p.full_name, "Phone": p.phone or "—",
            "Email": p.email or "—", "DOB": p.dob or "—",
            "Gender": {"M": "Male", "F": "Female"}.get(p.gender or "", p.gender or "—"),
        }.items()]
        print("Patient Record (Supabase → patients table)")
        display(pd.DataFrame(rows).style.hide(axis="index"))
        bks = (
            session.query(Booking, Doctor, Location)
            .join(Doctor, Booking.doctor_id == Doctor.doctor_id)
            .join(Location, Booking.location_id == Location.location_id)
            .filter(Booking.patient_id == p.patient_id)
            .order_by(Booking.start_at.desc()).limit(8).all()
        )
        if bks:
            bk_rows = []
            for bk, doc, loc in bks:
                spec = session.get(Specialty, doc.specialty_id) if doc.specialty_id else None
                bk_rows.append({
                    "Date": datetime.fromtimestamp(bk.start_at).strftime("%Y-%m-%d %H:%M"),
                    "Doctor": f"Dr. {doc.full_name}", "Specialty": spec.name if spec else "—",
                    "Location": loc.name, "Status": bk.status,
                })
            print(f"\nBookings ({len(bk_rows)} records — Supabase → bookings table)")
            display(pd.DataFrame(bk_rows))
    finally:
        session.close()


# ── Identify user ──────────────────────────────────────────────
greeting = "Hi there! I'm Anushka, and my mobile number is +94 78 10 30 736."
USER_ID = extract_phone(greeting)
SESSION_ID = "nb01-demo"

logger.success(f"USER_ID  = {USER_ID}")
logger.info(f"SESSION  = {SESSION_ID}")
show_patient(USER_ID)

---
## Section 3 — The Routing Engine

The **QueryRouter** sends the user message + memory context to `gpt-4o-mini` and receives a structured `RouteDecision`:

```python
@dataclass
class RouteDecision:
    route:      str    # crm | rag | web_search | direct
    confidence: float  # 0.0 – 1.0
    reasoning:  str
    action:     Optional[str]       # e.g. lookup_patient, search_doctors
    params:     Dict[str, Any]      # extracted parameters for the tool
```

Each route dispatches to a different tool:

| Route | Tool | Backend |
|---|---|---|
| `direct` | None — LLM answers directly | — |
| `crm` | `CRMTool` | SQLAlchemy → Supabase PostgreSQL |
| `rag` | `RAGTool` | CAG cache (Qdrant) → CRAG → Qdrant KB |
| `web_search` | `WebSearchTool` | Tavily API |

### 3a · `direct` Route — Greeting (No Tool)
Conversational queries are answered by the LLM directly. No tool is dispatched. Memory context (if any) is still injected.

In [ ]:
resp = agent.chat(user_message=greeting, user_id=USER_ID, session_id=SESSION_ID)
show_response(resp, "DIRECT route — greeting")

### 3b · `crm` Route — Patient Lookup
Appointment queries route to the CRM tool which queries Supabase PostgreSQL via SQLAlchemy.

In [ ]:
resp = agent.chat(
    user_message="Can you check my upcoming appointments? My phone number is 078 103 0736.",
    user_id=USER_ID, session_id=SESSION_ID,
)
show_response(resp, "CRM route — lookup_patient")

### 3c · `crm` Route — Doctor Search
Specialty search queries also route to CRM. The router extracts `specialty` as a structured param.

In [ ]:
resp = agent.chat(
    user_message="I need to see a cardiologist at Nawaloka Hospital. Who's available?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show_response(resp, "CRM route — search_doctors")

# Verify against actual DB
session = _crm_session()
try:
    docs = (
        session.query(Doctor, Specialty)
        .join(Specialty, Doctor.specialty_id == Specialty.specialty_id)
        .filter(Specialty.name.ilike("%cardiol%"), Doctor.active == 1)
        .limit(5).all()
    )
    if docs:
        df = pd.DataFrame([{"Doctor": f"Dr. {d.full_name}", "Specialty": s.name, "Phone": d.phone or "—"}
                           for d, s in docs])
        print("\nCardiologists in DB (Supabase → doctors table):")
        display(df)
finally:
    session.close()

### 3d · `rag` Route — Internal Knowledge Base

Hospital policy questions route to the **RAG pipeline**:

```
Query → CAGCache (Qdrant KNN-1, threshold=0.90)
          │ MISS
          ▼
        CRAGService (confidence gating)
          ├─ Retrieve k=4 → LLM self-evaluate → confidence < 0.6?
          │   └─ Yes → retry with k=8
          └─ Generate answer → cache result
```

The CAG cache is pre-warmed with 65+ FAQs from `config/faqs.yaml`, so common questions are answered instantly.

In [ ]:
# First call → cache MISS → Qdrant KB retrieval + LLM → cached
resp = agent.chat(
    user_message="What are the hospital's infection control and sterilization guidelines?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show_response(resp, "RAG route — Qdrant KB (CAG + CRAG)")

In [ ]:
# Second call — same semantic intent → cache HIT (instant)
resp2 = agent.chat(
    user_message="Tell me about PPE and sterilization protocols at the hospital.",
    user_id=USER_ID, session_id=SESSION_ID,
)
show_response(resp2, "RAG route — CAG cache HIT (semantic match, instant answer)")

# Inspect cache stats
stats = agent.rag_tool.cache_stats()
print(f"\nCAG Cache stats:")
print(f"  Entries    : {stats['total_cached']}")
print(f"  Threshold  : {stats['similarity_threshold']}")
print(f"  Backend    : {stats['backend']}")
print(f"  Collection : {stats['collection']}")
print(f"  TTL        : {stats['ttl_seconds']}s")
print(f"  Available  : {stats['available']}")

### 3e · `web_search` Route — Real-Time Information
For questions needing live data (current hours, events, news), the router dispatches to **Tavily Web Search**.

In [ ]:
resp = agent.chat(
    user_message="What are the current visiting hours at Nawaloka Hospital?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show_response(resp, "WEB SEARCH route — Tavily real-time")

---
## Section 4 — Memory System: 4 Layers

The agent has 4 memory types working together:

| Layer | What It Stores | Retrieval | Backend |
|---|---|---|---|
| **Short-Term** | Recent conversation turns | Last N turns (recency) | Supabase `st_turns` |
| **Long-Term Semantic** | Distilled facts & preferences | Cosine similarity (pgvector) | Supabase `mem_facts` |
| **Episodic** | Full conversation sessions | Cosine similarity on summary | Supabase `mem_episodes` |
| **Procedural** | Step-by-step workflows | Cosine similarity | Supabase `mem_procedures` |

### 4a · Short-Term Memory — Conversation Ring Buffer

- **Capacity**: Ring buffer — last `ST_MAX_TURNS=30` turns per user/session
- **TTL**: 24 hours (configurable)
- **Purpose**: Conversational continuity within and across sessions

In [ ]:
MEM_USER    = USER_ID
MEM_SESSION = "nb01-memory-demo"

# Build a realistic conversation about Anushka's medical profile
conversation = [
    ("user",      "Hi, I'm Anushka. I have a cardiac stress test coming up."),
    ("assistant", "Hello Anushka! I can see that in your records. How can I help you prepare?"),
    ("user",      "From now on, remember that I take atenolol 50mg every morning for blood pressure."),
    ("assistant", "Got it! I'll remember your atenolol 50mg daily medication schedule."),
    ("user",      "I'm allergic to penicillin — please always remember this."),
    ("assistant", "Noted! Penicillin allergy recorded. This is critical safety information."),
    ("user",      "Also remind me that I have a meniscus tear follow-up with orthopedics."),
    ("assistant", "Remembered! Orthopedics follow-up for the meniscus tear is noted."),
]

print(f"Storing {len(conversation)} turns in short-term memory...\n")
for role, content in conversation:
    turn = ConversationTurn(
        user_id=MEM_USER, session_id=MEM_SESSION,
        role=role, content=content, ts=time.time(),
    )
    st_store.append(turn, ST_MAX_TURNS, ST_TTL_SECONDS)
    emoji = "👤" if role == "user" else "🤖"
    print(f"  {emoji} [{role:9s}]: {content}")
    time.sleep(0.05)

logger.success(f"\n{len(conversation)} turns stored in Supabase (st_turns)")

In [ ]:
recent = st_store.recent(MEM_USER, MEM_SESSION, k=10)

print(f"Retrieved {len(recent)} turns from ST memory:\n")
for i, t in enumerate(recent, 1):
    emoji = "👤" if t.role == "user" else "🤖"
    age = f"{time.time() - t.ts:.0f}s ago"
    print(f"  {i}. {emoji} [{t.role:9s}] ({age}): {t.content}")

---
## Section 5 — Memory Distillation: LLM Extracts Long-Term Facts

The **MemoryDistiller** watches the conversation and extracts important facts when:
- Turn count ≥ 5, OR
- Conversation contains keywords: `remember`, `from now on`, `remind me`, `always`, `never`

**Process:**
```
Conversation Turns
    → LLM (llama-3.1-8b-instant / Groq)  ← ultra-fast extraction
    → JSON array of {text, tags}
    → score_memory_fact()  (0.5×recency + 0.3×repetition + 0.2×explicitness)
    → dedupe_facts()  (NumPy cosine matrix within batch, threshold=0.85)
    → LongTermMemoryStore.upsert()  (pgvector dedup, threshold=0.92)
```

In [ ]:
should = distiller.should_distill(recent)

print(f"Should distill? {should}")
print(f"\nTrigger analysis:")
print(f"  Turn count    : {len(recent)} (threshold: 5)")
for kw in ["remember", "from now on", "remind me", "always", "never"]:
    hit = any(kw in t.content.lower() for t in recent)
    print(f"  '{kw}'  : {hit}")

In [ ]:
print("Running memory distillation (LLM call)...\n")
facts = distiller.distill(MEM_USER, recent)

logger.success(f"Extracted {len(facts)} long-term facts:")
for i, f in enumerate(facts, 1):
    print(f"\n  {i}. {f.text}")
    print(f"     Score : {f.score:.3f}")
    print(f"     Tags  : {', '.join(f.tags)}")

In [ ]:
# Verify facts are semantically searchable via pgvector
test_queries = ["blood pressure medication", "drug allergies", "orthopedics follow-up"]

print("Semantic search over long-term memory (pgvector cosine):")
for q in test_queries:
    results = lt_store.query(MEM_USER, q, k=3, threshold=0.3)
    print(f"\n  Query: '{q}'  →  {len(results)} fact(s)")
    for r in results:
        print(f"    ↳ {r.text}  [score={r.score:.3f}]")

---
## Section 6 — Long-Term Semantic Recall: Token-Budgeted Retrieval

The **MemoryRecaller** combines ST and LT memories within a **token budget**:

```
max_tokens = 500
  ├─ Short-term budget : 60% = 300 tokens  (recent turns, max 4)
  └─ Long-term budget  : 40% = 200 tokens  (semantic facts, max 3)
```

LT facts are retrieved via **cosine similarity** against the user's query — so the most *relevant* facts surface, not just the most recent.

In [ ]:
query = "What medications does Anushka take and does she have any allergies?"

st_turns, lt_facts = recaller.recall(
    user_id=MEM_USER, session_id=MEM_SESSION,
    query=query, k_st=6, k_lt=5, max_tokens=500,
)

print(f"Recalled: {len(st_turns)} ST turns  +  {len(lt_facts)} LT facts")

st_tokens = sum(recaller.count_tokens(t.content) for t in st_turns)
lt_tokens = sum(recaller.count_tokens(f.text)    for f in lt_facts)
used      = st_tokens + lt_tokens
denom     = used if used > 0 else 1  # guard against zero-division

print(f"\nToken budget:")
print(f"  Total   : {used}/500")
print(f"  ST (60%): {st_tokens} tokens ({st_tokens/denom*100:.1f}% of used)")
print(f"  LT (40%): {lt_tokens} tokens ({lt_tokens/denom*100:.1f}% of used)")

# format_context only accepts st_turns; build LT section manually
st_context = recaller.format_context(st_turns)
if lt_facts:
    lt_lines = ["=== REMEMBERED FACTS ==="]
    for i, f in enumerate(lt_facts, 1):
        lt_lines.append(f"{i}. {f.text}  [{', '.join(f.tags)}]")
    lt_context = "\n".join(lt_lines)
else:
    lt_context = ""

full_context = "\n".join(part for part in [st_context, lt_context] if part)

print(f"\nMemory context passed to LLM:")
print("-" * 60)
print(full_context)

---
## Section 7 — Episodic Memory: Full Conversation Snapshots

Episodic memory stores **complete sessions** — not just extracted facts.

- **Semantic vs Episodic**: `"User takes atenolol 50mg"` (semantic fact) vs the full 8-turn conversation where this was discussed, with an LLM-generated summary (episode).
- **Use case**: *"What did we discuss last Tuesday?"* → episodic search returns the session.

In [ ]:
episodic_store = EpisodicMemoryStore(embedder)

# Create episode from the conversation turns (LLM generates summary)
episode = create_episode_from_turns(
    user_id=MEM_USER, session_id=MEM_SESSION,
    turns=recent, llm=llm,
)

print(f"Episode created:")
print(f"  Session : {episode.session_id}")
print(f"  Turns   : {episode.turn_count}")
print(f"  Topics  : {', '.join(episode.topic_tags)}")
print(f"  Summary : {episode.summary}")

episodic_store.store_episode(episode)
logger.success("Episode stored in Supabase pgvector")

# Query episodic memory
for q in ["medication discussion", "allergy information"]:
    eps = episodic_store.query_episodes(MEM_USER, q, k=2, threshold=0.3)
    print(f"\n  Query '{q}' → {len(eps)} episode(s)")
    for ep in eps:
        print(f"    Topics: {', '.join(ep.topic_tags)} | {ep.turn_count} turns")
        print(f"    Summary: {ep.summary[:120]}...")

---
## Section 8 — Procedural Memory: Workflow Guidance

Procedural memory stores **step-by-step workflows** pre-seeded by an admin. When a user query matches a procedure, the agent retrieves and follows the steps — ensuring consistency and explainability.

Example: `"Book an appointment"` → retrieves `book_new_appointment` procedure → follows 8 steps.

In [ ]:
from memory.prompts import format_procedures

proc_store = ProceduralMemoryStore()
query      = "I want to book an appointment with a cardiologist"

procedures = proc_store.query_procedures(query, top_k=2, threshold=0.3)

if procedures:
    for p in procedures:
        logger.success(f"Matched: {p.name}  (similarity={p.similarity:.2f})")
        print(f"  Category    : {p.category}")
        print(f"  Description : {p.description}")
        print(f"  Steps       : {len(p.steps)}")
    print("\nFormatted for agent prompt:")
    print("=" * 60)
    print(format_procedures(procedures))
else:
    logger.warning("No procedures found. Run: python scripts/seed_procedures.py")

---
## Section 9 — Without vs. With Memory: The Impact of Context

This is the core demonstration of why memory matters. The **same LLM** gives:
- A **generic, useless** answer without memory
- A **personalised, accurate** answer with recalled context

Same model. Same query. Different context → fundamentally different response.

In [ ]:
compare_query = "What medications does Anushka take and does she have any allergies?"

# ── Without memory ─────────────────────────────────────────────
print("=" * 72)
print("WITHOUT memory (raw LLM — no context):")
print("=" * 72)
baseline = llm.invoke(compare_query)
baseline_text = baseline.content if hasattr(baseline, "content") else str(baseline)
print(baseline_text)

# ── Rebuild full_context if this cell is run independently ─────
# (st_turns / lt_facts come from the recall cell above)
st_ctx = recaller.format_context(st_turns)
if lt_facts:
    lt_lines = ["=== REMEMBERED FACTS ==="]
    for i, f in enumerate(lt_facts, 1):
        lt_lines.append(f"{i}. {f.text}  [{', '.join(f.tags)}]")
    lt_ctx = "\n".join(lt_lines)
else:
    lt_ctx = ""
full_context = "\n".join(part for part in [st_ctx, lt_ctx] if part)
injected_tokens = used  # set in the recall cell above

prompt = f"{full_context}\n\nUSER QUERY: {compare_query}\n\nAnswer based on the above:"

print("\n" + "=" * 72)
print(f"WITH memory ({injected_tokens} tokens of context injected):")
print("=" * 72)
memory_resp = llm.invoke(prompt)
memory_text = memory_resp.content if hasattr(memory_resp, "content") else str(memory_resp)
print(memory_text)

print("\n" + "=" * 72)
logger.error("WITHOUT memory: LLM refuses — no context about this user")
logger.success(f"WITH memory ({injected_tokens} tokens): LLM gives precise, personalised answer")

---
## Section 10 — Multi-Turn: Progressive Memory Building

Watch memory **accumulate** across turns. Each turn adds to short-term memory. The distiller extracts long-term facts. By the end, the agent knows the user deeply — and the memory persists **across sessions** (LT facts survive in pgvector).

In [ ]:
MULTI_SESSION = "nb01-multiturn"
first_msg = "Hi, I'm Anushka. My mobile is 078 103 0736. I have a cardiac stress test coming up."
MULTI_USER = extract_phone(first_msg)

messages = [
    first_msg,
    "I take atenolol 50mg every morning for blood pressure. Please remember this.",
    "I'm allergic to penicillin — very important, always remember!",
    "Can you find me a cardiologist?",
    "What is the medication administration policy at the hospital?",
    "What do you remember about my health profile and medications?",
]

print("Multi-Turn Conversation — Progressive Memory Building")
print("=" * 72)

for i, msg in enumerate(messages, 1):
    print(f"\n{'─' * 72}")
    print(f"Turn {i}: {msg}")
    print("─" * 72)

    resp = agent.chat(user_message=msg, user_id=MULTI_USER, session_id=MULTI_SESSION)

    ctx_lines = len(resp.memory_context.strip().split("\n")) if resp.memory_context.strip() else 0
    print(f"  Route          : {resp.route}" + (f" / {resp.action}" if resp.action else ""))
    print(f"  Memory context : {ctx_lines} lines (grows each turn)")
    print(f"  Latency        : {resp.latency_ms}ms")
    print(f"  Answer         : {resp.answer[:300]}{'...' if len(resp.answer) > 300 else ''}")

print(f"\n{'=' * 72}")
logger.success("Multi-turn complete. Memory context grew from 0 → N lines across turns.")
logger.info("Long-term facts now persisted in pgvector — will survive across sessions.")

---
## Section 11 — Observability with LangFuse

Every `.chat()` call creates a **LangFuse trace** with nested spans:

```
agent_chat  (trace)
  ├─ memory_recall        (span)       ← ST + LT retrieval time
  ├─ router               (generation) ← gpt-4o-mini tokens + cost
  ├─ tool_dispatch        (span)
  │    └─ crm | rag | web_search  (span)
  ├─ synthesiser          (generation) ← gemini-2.5-flash tokens + cost
  ├─ memory_store         (span)
  └─ memory_distill       (span)
       └─ distill_facts   (generation) ← llama-3.1-8b tokens + cost
```

Open your **LangFuse dashboard** → Traces to see per-step latency, token usage, cost, and I/O for every call.

In [ ]:
from infrastructure.observability import flush

flush()  # Push pending events to LangFuse

host = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
print(f"LangFuse dashboard: {host}")
print("  Traces → filter by tag 'agent' → see full pipeline waterfall.")
print("  Each trace: recall → route → dispatch → synthesise → store → distill")

---
## Summary

| Component | Role | Key File |
|---|---|---|
| **QueryRouter** | LLM classifies intent → `crm \| rag \| web_search \| direct` | `agents/router.py` |
| **CRMTool** | Patient lookup, doctor search, booking CRUD | `agents/tools/crm_tool.py` |
| **RAGTool** | CAG cache → CRAG → Qdrant KB | `agents/tools/rag_tool.py` |
| **WebSearchTool** | Tavily real-time search | `agents/tools/web_search_tool.py` |
| **ShortTermStore** | Ring buffer (30 turns, 24h TTL) | `memory/st_store.py` |
| **LongTermStore** | Semantic facts (pgvector, cosine KNN) | `memory/lt_store.py` |
| **EpisodicStore** | Full session snapshots (pgvector) | `memory/episodic_store.py` |
| **ProceduralStore** | Workflow steps (pgvector) | `memory/procedural_store.py` |
| **MemoryDistiller** | LLM extracts facts from turns | `memory/memory_ops.py` |
| **MemoryRecaller** | Token-budgeted hybrid recall | `memory/memory_ops.py` |
| **LangFuse** | Traces every step (latency, tokens, cost) | `infrastructure/observability.py` |

---

**Next →** Notebook 02 replaces this entire imperative pipeline with a **LangGraph StateGraph**, introducing the Supervisor-Worker pattern, typed shared state, and persona-specialized sub-agents.